In [1]:

# ==============================================================================
# GERADOR SINTÉTICO DE DADOS - SAC MÓVEIS RESIDENCIAIS
# ==============================================================================
import pandas as pd
import random

templates = {
    'vendas': {
        's': ['', 'Olá', 'Bom dia', 'Gostaria de saber', 'Por favor'],
        'a': ['quero comprar', 'qual o preco do', 'tem cupom para', 'como faco para adquirir', 'desejo orcamento de'],
        'o': ['sofa retratil 3 lugares', 'conjunto de mesa de jantar', 'guarda roupa casal', 'painel para tv', 'colchao queen size']
    },
    'suporte': {
        's': ['', 'Oi', 'Preciso de ajuda', 'Por gentileza', 'Socorro'],
        'a': ['como montar o', 'onde baixo o manual do', 'estou com duvida no', 'veio faltando parafuso no', 'preciso de assistencia para'],
        'o': ['armario de cozinha', 'rack da sala', 'berco do bebe', 'esquema de montagem', 'manual da estante']
    },
    'trocas_devolucoes': {
        's': ['', 'Olá', 'Por favor', 'Gostaria de solicitar', 'Quero abrir'],
        'a': ['preciso trocar o', 'quero devolver a', 'como solicito o estorno do', 'desejo solicitar a troca da', 'como funciona a devolucao do'],
        'o': ['produto com defeito', 'mesa que veio arranhada', 'cadeira no prazo de 7 dias', 'pedido cancelado', 'item com avaria']
    },
    'reclamacoes': {
        's': ['', 'Urgente', 'Pessimo atendimento', 'Absurdo', 'Quero registrar'],
        'a': ['estou indignado com o', 'quero fazer uma queixa do', 'estou reclamando do', 'produto veio quebrado e o', 'atendimento horrivel do'],
        'o': ['atraso na minha entrega', 'servico de montagem', 'sac que nao responde', 'pos venda da loja', 'estado do meu movel']
    },
    'logistica_entregas': {
        's': ['', 'Olá', 'Bom dia', 'Por gentileza', 'Preciso saber'],
        'a': ['onde esta o meu', 'qual o prazo de entrega do', 'como rastreio a', 'qual a transportadora do', 'quando chega o'],
        'o': ['meu pedido', 'codigo de rastreamento', 'movel comprado', 'status do envio', 'agendamento da entrega']
    }
}

amostras = []
random.seed(42)

for intencao, comp in templates.items():
    for _ in range(20):  # Total: 100 amostras (20 por classe)
        s = random.choice(comp['s'])
        a = random.choice(comp['a'])
        o = random.choice(comp['o'])
        frase = f"{s} {a} {o}".strip().capitalize()
        amostras.append({'texto': frase, 'intencao': intencao})

df_moveis = pd.DataFrame(amostras)
df_moveis.to_csv('dataset_moveis_100.csv', index=False, encoding='utf-8')

print(" Dataset 'dataset_moveis_100.csv' criado com 100 frases distribuidas em 5 intencoes!")

 Dataset 'dataset_moveis_100.csv' criado com 100 frases distribuidas em 5 intencoes!


In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# 1. Carregar dataset do CSV
df = pd.read_csv('dataset_moveis_100.csv')

# 2. Divisão Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(
    df['texto'],
    df['intencao'],
    test_size=0.20,
    random_state=42,
    stratify=df['intencao']
)

# TODO 1: Criar Pipeline
pipeline_knn = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', KNeighborsClassifier(
        n_neighbors=4,
        metric='cosine'
    ))
])

# TODO 2: Treinar Pipeline
pipeline_knn.fit(X_train, y_train)

# TODO 3: Fazer previsões
y_pred = pipeline_knn.predict(X_test)

print("\n=== RELATÓRIO DE CLASSIFICAÇÃO ===")
print(classification_report(y_test, y_pred))

print("\n=== MATRIZ DE CONFUSÃO ===")
print(confusion_matrix(y_test, y_pred))

LIMIAR_CONFIANCA = 0.60

print("\n=== INICIANDO BATERIA DE TESTES (10 INPUTS OBRIGATÓRIOS) ===")

for i in range(1, 11):
    print(f"\n[Teste {i}/10]")

    # TODO 4: Receber frase
    frase = input("Digite a frase do cliente: ").strip()

    # TODO 5: Probabilidades e intenção
    probs = pipeline_knn.predict_proba([frase])[0]

    maior_prob = np.max(probs)

    intencao = pipeline_knn.predict([frase])[0]

    # TODO 6: Regra de decisão
    if maior_prob >= LIMIAR_CONFIANCA:
        print(f"Intenção identificada: {intencao}")
        print(f"Confiança: {maior_prob * 100:.2f}%")
    else:
        print("Fallback ativado.")
        print("Não consegui identificar sua solicitação com segurança.")
        print("Encaminhando para a equipe de atendimento humano.")
        print(f"Confiança obtida: {maior_prob * 100:.2f}%")


=== RELATÓRIO DE CLASSIFICAÇÃO ===
                    precision    recall  f1-score   support

logistica_entregas       0.80      1.00      0.89         4
       reclamacoes       1.00      1.00      1.00         4
           suporte       1.00      1.00      1.00         4
 trocas_devolucoes       1.00      1.00      1.00         4
            vendas       1.00      0.75      0.86         4

          accuracy                           0.95        20
         macro avg       0.96      0.95      0.95        20
      weighted avg       0.96      0.95      0.95        20


=== MATRIZ DE CONFUSÃO ===
[[4 0 0 0 0]
 [0 4 0 0 0]
 [0 0 4 0 0]
 [0 0 0 4 0]
 [1 0 0 0 3]]

=== INICIANDO BATERIA DE TESTES (10 INPUTS OBRIGATÓRIOS) ===

[Teste 1/10]
Digite a frase do cliente: qual o preço do sofá?
Intenção identificada: logistica_entregas
Confiança: 100.000%

[Teste 2/10]
Digite a frase do cliente: desejo orcamento do sofa
Intenção identificada: vendas
Confiança: 100.000%

[Teste 3/10]
Digite a f

In [6]:
# ATIVIDADE 2: CHATBOT VERSÃO 2 — DECISION TREE

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split


# ==============================================================================
# 1. CARREGAR O DATASET
# ==============================================================================

df = pd.read_csv('dataset_moveis_100.csv')


# ==============================================================================
# 2. DIVIDIR OS DADOS EM TREINO E TESTE
# ==============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    df['texto'],
    df['intencao'],
    test_size=0.20,
    random_state=42,
    stratify=df['intencao']
)


# ==============================================================================
# 3. CRIAR A PIPELINE
# ==============================================================================

pipeline_tree = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', DecisionTreeClassifier(random_state=42))
])


# ==============================================================================
# 4. TREINAR O MODELO
# ==============================================================================

pipeline_tree.fit(X_train, y_train)


# ==============================================================================
# 5. FAZER PREVISÕES NOS DADOS DE TESTE
# ==============================================================================

y_pred = pipeline_tree.predict(X_test)


# ==============================================================================
# 6. EXIBIR OS RESULTADOS
# ==============================================================================

print("\n=== RELATÓRIO DE CLASSIFICAÇÃO ===")

print(classification_report(y_test, y_pred))


print("\n=== MATRIZ DE CONFUSÃO ===")

print(confusion_matrix(y_test, y_pred))


# ==============================================================================
# 7. DEFINIR O LIMIAR DE CONFIANÇA
# ==============================================================================

LIMIAR_CONFIANCA = 0.60


# ==============================================================================
# 8. REALIZAR 8 TESTES MANUAIS
# ==============================================================================

print("\n=== INICIANDO BATERIA DE TESTES (8 INPUTS OBRIGATÓRIOS) ===")


for i in range(1, 9):

    print(f"\n[Teste {i}/8]")


    # Receber a frase do usuário
    frase = input("Digite a frase do cliente: ").strip()


    # Obter as probabilidades de cada intenção
    probs = pipeline_tree.predict_proba([frase])[0]


    # Encontrar a maior probabilidade
    maior_prob = np.max(probs)


    # Obter a intenção prevista
    intencao = pipeline_tree.predict([frase])[0]


    # Verificar o nível de confiança
    if maior_prob >= LIMIAR_CONFIANCA:

        print(f"Intenção identificada: {intencao}")
        print(f"Confiança: {maior_prob * 100:.2f}%")

    else:

        print("\nDesculpe, não entendi sua solicitação.")
        print("Encaminhando você para um atendente humano...")
        print(f"Confiança obtida: {maior_prob * 100:.2f}%")



=== RELATÓRIO DE CLASSIFICAÇÃO ===
                    precision    recall  f1-score   support

logistica_entregas       0.80      1.00      0.89         4
       reclamacoes       1.00      0.75      0.86         4
           suporte       0.80      1.00      0.89         4
 trocas_devolucoes       1.00      0.75      0.86         4
            vendas       1.00      1.00      1.00         4

          accuracy                           0.90        20
         macro avg       0.92      0.90      0.90        20
      weighted avg       0.92      0.90      0.90        20


=== MATRIZ DE CONFUSÃO ===
[[4 0 0 0 0]
 [1 3 0 0 0]
 [0 0 4 0 0]
 [0 0 1 3 0]
 [0 0 0 0 4]]

=== INICIANDO BATERIA DE TESTES (8 INPUTS OBRIGATÓRIOS) ===

[Teste 1/8]
Digite a frase do cliente: quero comprar um sofá
Intenção identificada: vendas
Confiança: 100.00%

[Teste 2/8]
Digite a frase do cliente: quero falar com o gerente
Intenção identificada: trocas_devolucoes
Confiança: 100.00%

[Teste 3/8]
Digite a frase d